In [28]:
import pandas as pd
import numpy as np
from pyfixest.estimation import feols
from sklearn.preprocessing import StandardScaler

In [29]:
df = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\all_data_final_with_IMR_rename.csv')
# 处理缺失值
required_cols = ['ExcellenceRate','PassRate', 'r1h','rfh', 'r3h', 'FluctMonth', 'FluctIntra', 'FluctTenDays','RainHistVol','HeavyRain','LowRain','ExtreFluctMonth',\
                 'SchType', 'SchLocation', 'Gender', 'IMR','Grade', 'Date', 'RegionCode', 'District', 'SchYear']
df = df.dropna(subset=required_cols)
print("处理缺失值后，样本量:", len(df))


#核心自变量归一化
scaler = StandardScaler()
df['Normr1h'] = scaler.fit_transform(df[['r1h']])
df['Normr3h'] = scaler.fit_transform(df[['r3h']])
df['NormFluctMonth'] = scaler.fit_transform(df[['FluctMonth']])



#缩尾处理进行鲁棒性check
# 执行缩尾处理 (Winsorization)
# 理由：这能防止极端降水值过度左右回归系数，同时保留降雨分布的主要特征。
core_vars = ['r1h', 'r3h', 'FluctMonth']
for var in core_vars:
    lower_limit = df[var].quantile(0.01)#0.01 0.05
    upper_limit = df[var].quantile(0.99)#0.99 0.95
    lower_limit2 = df[var].quantile(0.05)#0.01 0.05
    upper_limit2 = df[var].quantile(0.95)#0.99 0.95

    df[f'NormWins{var}'] = df[var].clip(lower_limit, upper_limit)
    df[f'NormWins{var}2'] = df[var].clip(lower_limit2, upper_limit2)
for var in core_vars:
    df[f'NormWins{var}'] = scaler.fit_transform(df[[f'NormWins{var}']])
    df[f'NormWins{var}2'] = scaler.fit_transform(df[[f'NormWins{var}2']])

处理缺失值后，样本量: 64743


In [30]:
# 2. 计算 Pearson 相关系数矩阵
corr_matrix = df[['Normr1h','Normr3h','NormFluctMonth']].corr()

# 3. 打印相关系数矩阵
print("核心变量相关系数矩阵：")
print(corr_matrix)

核心变量相关系数矩阵：
                 Normr1h   Normr3h  NormFluctMonth
Normr1h         1.000000  0.493960       -0.248133
Normr3h         0.493960  1.000000        0.166068
NormFluctMonth -0.248133  0.166068        1.000000


# 一、主效应

In [31]:
#定义回归模型  回归公式结构： Y ~ X1 + X2 + ... | FE1 + FE2 + ...

# A. 因变量 (Y)：PassRate,ExcellenceRate (分别回归)
# B. 核心自变量 (Rainfall): Normr1h  (过去一个月的降雨量)
# C. 其他控制变量 (School/Student Charateristics)：
#    - SchType, SchLocation
#    - Gender,C(Grade)    Grade是分类变量但作为控制变量放入前半部分，使用 C(Grade)
# D. 固定效应 (Fixed Effects)：
#    - RegionCode (二级行政区编码)
#    - Date (日期)
# E. 选择效应:
#    - IMR作为控制变量纳入模型

# 构建回归公式
fe_part = 'District + Date'
fe_part2 = 'RegionCode + Date'
fe_part3 = 'District^SchYear+Date'
fe_part4 = 'District^Date'

control_vars = 'SchType + SchLocation + Gender + IMR + C(Grade)'
control_vars_orig = 'SchType + SchLocation + Gender + C(Grade)'

#主模型
formula_orig_1 = f"PassRate ~ Normr1h + Normr3h + NormFluctMonth + {control_vars_orig}"
formula_orig_2 = f"ExcellenceRate ~ Normr1h + Normr3h + NormFluctMonth + {control_vars_orig}"
formula_orig_3 = f"PassRate ~ Normr1h + Normr3h + NormFluctMonth + {control_vars_orig} | {fe_part}"
formula_orig_4 = f"ExcellenceRate ~ Normr1h + Normr3h + NormFluctMonth + {control_vars_orig} | {fe_part}"

formula0_1 = f"PassRate ~ Normr1h + Normr3h + NormFluctMonth + {control_vars} | {fe_part}"
formula0_2 = f"ExcellenceRate ~ Normr1h + Normr3h + NormFluctMonth + {control_vars} | {fe_part}"


#鲁棒性检验
RBformula1_1 = f"PassRate ~ NormWinsr1h + NormWinsr3h + NormWinsFluctMonth + {control_vars} | {fe_part}"
RBformula1_2 = f"ExcellenceRate ~ NormWinsr1h + NormWinsr3h + NormWinsFluctMonth + {control_vars} | {fe_part}"
RBformula1_3 = f"PassRate ~ NormWinsr1h2 + NormWinsr3h2 + NormWinsFluctMonth2 + {control_vars} | {fe_part}"
RBformula1_4 = f"ExcellenceRate ~ NormWinsr1h2 + NormWinsr3h2 + NormWinsFluctMonth2 + {control_vars} | {fe_part}"


RBformula2_1 = f"PassRate ~ Normr1h + Normr3h + NormFluctMonth + {control_vars} | {fe_part2}"
RBformula2_2 = f"ExcellenceRate ~ Normr1h + Normr3h + NormFluctMonth + {control_vars} | {fe_part2}"
RBformula3_1 = f"PassRate ~ Normr1h + Normr3h + NormFluctMonth + {control_vars} | {fe_part3}"
RBformula3_2 = f"ExcellenceRate ~ Normr1h + Normr3h + NormFluctMonth + {control_vars} | {fe_part3}"
RBformula4_1 = f"PassRate ~ Normr1h + Normr3h + NormFluctMonth + {control_vars} | {fe_part4}"
RBformula4_2 = f"ExcellenceRate ~ Normr1h + Normr3h + NormFluctMonth + {control_vars} | {fe_part4}"

In [32]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'RegionCode'}
# 双向聚类标准误，同时聚类到地区和时间：vcov={'CRV3': 'RegionCode + Date'} 

try:
    model_result = feols(fml=formula_orig_1, data=df, vcov= 'hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())

except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: PassRate, Fixed effects: 0
Inference:  hetero
Observations:  64743

| Coefficient     |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:----------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept       |      0.791 |        0.003 |   263.752 |      0.000 |  0.785 |   0.797 |
| Normr1h         |     -0.012 |        0.001 |    -9.960 |      0.000 | -0.014 |  -0.010 |
| Normr3h         |     -0.006 |        0.001 |    -4.943 |      0.000 | -0.008 |  -0.004 |
| NormFluctMonth  |     -0.024 |        0.001 |   -20.698 |      0.000 | -0.027 |  -0.022 |
| SchType         |      0.073 |        0.004 |    19.302 |      0.000 |  0.066 |   0.081 |
| SchLocation     |      0.000 |        0.002 |     0.136 |      0.892 | -0.004 |   0.004 |
| Gender          |     -0.011 |        0.002 |    -6.104 |      0.000 | -0.015 |  -0.008 |
| C(Grade)[T.CE2] |      0.012 |        0.

In [33]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'RegionCode'}
# 双向聚类标准误，同时聚类到地区和时间：vcov={'CRV3': 'RegionCode + Date'} 

try:
    model_result = feols(fml=formula_orig_2, data=df, vcov= 'hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())

except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: ExcellenceRate, Fixed effects: 0
Inference:  hetero
Observations:  64743

| Coefficient     |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:----------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept       |      0.516 |        0.003 |   151.008 |      0.000 |  0.509 |   0.522 |
| Normr1h         |     -0.009 |        0.001 |    -6.939 |      0.000 | -0.012 |  -0.007 |
| Normr3h         |     -0.012 |        0.001 |    -8.890 |      0.000 | -0.014 |  -0.009 |
| NormFluctMonth  |     -0.023 |        0.001 |   -19.673 |      0.000 | -0.025 |  -0.021 |
| SchType         |      0.064 |        0.005 |    13.204 |      0.000 |  0.054 |   0.073 |
| SchLocation     |      0.034 |        0.002 |    15.045 |      0.000 |  0.030 |   0.038 |
| Gender          |     -0.009 |        0.002 |    -4.514 |      0.000 | -0.013 |  -0.005 |
| C(Grade)[T.CE2] |      0.002 |    

In [34]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'RegionCode'}
# 双向聚类标准误，同时聚类到地区和时间：vcov={'CRV3': 'RegionCode + Date'} 

try:
    model_result = feols(fml=formula_orig_3, data=df, vcov= 'hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())

except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: PassRate, Fixed effects: District+Date
Inference:  hetero
Observations:  64743

| Coefficient     |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:----------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Normr1h         |     -0.005 |        0.002 |    -1.904 |      0.057 | -0.009 |   0.000 |
| Normr3h         |      0.019 |        0.003 |     6.326 |      0.000 |  0.013 |   0.025 |
| NormFluctMonth  |     -0.017 |        0.002 |   -11.032 |      0.000 | -0.020 |  -0.014 |
| SchType         |      0.058 |        0.004 |    14.284 |      0.000 |  0.050 |   0.066 |
| SchLocation     |     -0.009 |        0.002 |    -4.125 |      0.000 | -0.013 |  -0.005 |
| Gender          |     -0.011 |        0.002 |    -6.211 |      0.000 | -0.015 |  -0.008 |
| C(Grade)[T.CE2] |      0.011 |        0.004 |     2.936 |      0.003 |  0.004 |   0.018 |
| C(Grade)[T.CM1] |      0.008

In [35]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'RegionCode'}
# 双向聚类标准误，同时聚类到地区和时间：vcov={'CRV3': 'RegionCode + Date'} 

try:
    model_result = feols(fml=formula_orig_4, data=df, vcov= 'hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())

except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: ExcellenceRate, Fixed effects: District+Date
Inference:  hetero
Observations:  64743

| Coefficient     |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:----------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Normr1h         |     -0.016 |        0.003 |    -5.979 |      0.000 | -0.021 |  -0.010 |
| Normr3h         |      0.028 |        0.003 |     8.457 |      0.000 |  0.022 |   0.035 |
| NormFluctMonth  |     -0.014 |        0.002 |    -9.182 |      0.000 | -0.018 |  -0.011 |
| SchType         |      0.037 |        0.005 |     7.294 |      0.000 |  0.027 |   0.047 |
| SchLocation     |      0.020 |        0.002 |     8.673 |      0.000 |  0.015 |   0.024 |
| Gender          |     -0.009 |        0.002 |    -4.621 |      0.000 | -0.013 |  -0.005 |
| C(Grade)[T.CE2] |      0.001 |        0.004 |     0.330 |      0.741 | -0.007 |   0.010 |
| C(Grade)[T.CM1] |     

In [36]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'RegionCode'}
# 双向聚类标准误，同时聚类到地区和时间：vcov={'CRV3': 'RegionCode + Date'} 

try:
    model_result = feols(fml=formula0_1, data=df, vcov= 'hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())

except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: PassRate, Fixed effects: District+Date
Inference:  hetero
Observations:  64743

| Coefficient     |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:----------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Normr1h         |     -0.005 |        0.002 |    -2.207 |      0.027 | -0.010 |  -0.001 |
| Normr3h         |      0.020 |        0.003 |     6.595 |      0.000 |  0.014 |   0.026 |
| NormFluctMonth  |     -0.018 |        0.002 |   -11.737 |      0.000 | -0.021 |  -0.015 |
| SchType         |      0.066 |        0.004 |    15.911 |      0.000 |  0.058 |   0.074 |
| SchLocation     |     -0.010 |        0.002 |    -4.640 |      0.000 | -0.014 |  -0.006 |
| Gender          |     -0.011 |        0.002 |    -6.214 |      0.000 | -0.015 |  -0.008 |
| IMR             |     -0.017 |        0.002 |    -8.999 |      0.000 | -0.021 |  -0.013 |
| C(Grade)[T.CE2] |      0.011

In [37]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'RegionCode'}
# 双向聚类标准误，同时聚类到地区和时间：vcov={'CRV3': 'RegionCode + Date'} 

try:
    model_result = feols(fml=formula0_2, data=df, vcov= 'hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())

except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: ExcellenceRate, Fixed effects: District+Date
Inference:  hetero
Observations:  64743

| Coefficient     |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:----------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Normr1h         |     -0.016 |        0.003 |    -6.129 |      0.000 | -0.021 |  -0.011 |
| Normr3h         |      0.029 |        0.003 |     8.593 |      0.000 |  0.022 |   0.035 |
| NormFluctMonth  |     -0.015 |        0.002 |    -9.549 |      0.000 | -0.018 |  -0.012 |
| SchType         |      0.042 |        0.005 |     8.024 |      0.000 |  0.031 |   0.052 |
| SchLocation     |      0.019 |        0.002 |     8.412 |      0.000 |  0.015 |   0.024 |
| Gender          |     -0.009 |        0.002 |    -4.621 |      0.000 | -0.013 |  -0.005 |
| IMR             |     -0.009 |        0.002 |    -4.532 |      0.000 | -0.014 |  -0.005 |
| C(Grade)[T.CE2] |     

In [38]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'RegionCode'}
# 双向聚类标准误，同时聚类到地区和时间：vcov={'CRV3': 'RegionCode + Date'} 

try:
    model_result = feols(fml=RBformula1_1, data=df, vcov= 'hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())

except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: PassRate, Fixed effects: District+Date
Inference:  hetero
Observations:  64743

| Coefficient        |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:-------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| NormWinsr1h        |     -0.005 |        0.002 |    -2.162 |      0.031 | -0.010 |  -0.000 |
| NormWinsr3h        |      0.020 |        0.003 |     6.596 |      0.000 |  0.014 |   0.026 |
| NormWinsFluctMonth |     -0.018 |        0.002 |   -11.772 |      0.000 | -0.022 |  -0.015 |
| SchType            |      0.066 |        0.004 |    15.913 |      0.000 |  0.058 |   0.074 |
| SchLocation        |     -0.010 |        0.002 |    -4.642 |      0.000 | -0.014 |  -0.006 |
| Gender             |     -0.011 |        0.002 |    -6.214 |      0.000 | -0.015 |  -0.008 |
| IMR                |     -0.017 |        0.002 |    -9.004 |      0.000 | -0.021 |  -0.013 |
| C

In [39]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'RegionCode'}
# 双向聚类标准误，同时聚类到地区和时间：vcov={'CRV3': 'RegionCode + Date'} 

try:
    model_result = feols(fml=RBformula1_2, data=df, vcov= 'hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())

except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: ExcellenceRate, Fixed effects: District+Date
Inference:  hetero
Observations:  64743

| Coefficient        |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:-------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| NormWinsr1h        |     -0.016 |        0.003 |    -6.036 |      0.000 | -0.021 |  -0.011 |
| NormWinsr3h        |      0.029 |        0.003 |     8.560 |      0.000 |  0.022 |   0.035 |
| NormWinsFluctMonth |     -0.015 |        0.002 |    -9.556 |      0.000 | -0.018 |  -0.012 |
| SchType            |      0.042 |        0.005 |     8.024 |      0.000 |  0.031 |   0.052 |
| SchLocation        |      0.019 |        0.002 |     8.410 |      0.000 |  0.015 |   0.024 |
| Gender             |     -0.009 |        0.002 |    -4.621 |      0.000 | -0.013 |  -0.005 |
| IMR                |     -0.009 |        0.002 |    -4.532 |      0.000 | -0.014 |  -0.005

In [40]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'RegionCode'}
# 双向聚类标准误，同时聚类到地区和时间：vcov={'CRV3': 'RegionCode + Date'} 

try:
    model_result = feols(fml=RBformula2_1, data=df, vcov= 'hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())

except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: PassRate, Fixed effects: RegionCode+Date
Inference:  hetero
Observations:  64743

| Coefficient     |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:----------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Normr1h         |     -0.007 |        0.002 |    -2.849 |      0.004 | -0.012 |  -0.002 |
| Normr3h         |      0.013 |        0.004 |     3.511 |      0.000 |  0.006 |   0.020 |
| NormFluctMonth  |     -0.020 |        0.002 |   -12.692 |      0.000 | -0.023 |  -0.017 |
| SchType         |      0.065 |        0.004 |    15.533 |      0.000 |  0.056 |   0.073 |
| SchLocation     |     -0.011 |        0.002 |    -5.375 |      0.000 | -0.015 |  -0.007 |
| Gender          |     -0.011 |        0.002 |    -6.217 |      0.000 | -0.015 |  -0.008 |
| IMR             |     -0.018 |        0.002 |    -8.601 |      0.000 | -0.022 |  -0.014 |
| C(Grade)[T.CE2] |      0.0

In [41]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'RegionCode'}
# 双向聚类标准误，同时聚类到地区和时间：vcov={'CRV3': 'RegionCode + Date'} 

try:
    model_result = feols(fml=RBformula2_2, data=df, vcov= 'hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())

except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: ExcellenceRate, Fixed effects: RegionCode+Date
Inference:  hetero
Observations:  64743

| Coefficient     |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:----------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Normr1h         |     -0.017 |        0.003 |    -6.301 |      0.000 | -0.022 |  -0.012 |
| Normr3h         |      0.025 |        0.004 |     6.301 |      0.000 |  0.017 |   0.032 |
| NormFluctMonth  |     -0.014 |        0.002 |    -8.565 |      0.000 | -0.017 |  -0.011 |
| SchType         |      0.040 |        0.005 |     7.709 |      0.000 |  0.030 |   0.050 |
| SchLocation     |      0.018 |        0.002 |     7.848 |      0.000 |  0.014 |   0.023 |
| Gender          |     -0.009 |        0.002 |    -4.620 |      0.000 | -0.013 |  -0.005 |
| IMR             |     -0.007 |        0.002 |    -2.951 |      0.003 | -0.011 |  -0.002 |
| C(Grade)[T.CE2] |   

In [42]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'RegionCode'}
# 双向聚类标准误，同时聚类到地区和时间：vcov={'CRV3': 'RegionCode + Date'} 

try:
    model_result = feols(fml=RBformula3_1, data=df, vcov= 'hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())

except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: PassRate, Fixed effects: District^SchYear+Date
Inference:  hetero
Observations:  64743

| Coefficient     |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:----------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Normr1h         |      0.002 |        0.002 |     1.001 |      0.317 | -0.002 |   0.007 |
| Normr3h         |      0.020 |        0.003 |     5.801 |      0.000 |  0.013 |   0.027 |
| NormFluctMonth  |     -0.001 |        0.002 |    -0.459 |      0.646 | -0.005 |   0.003 |
| SchType         |      0.067 |        0.004 |    16.242 |      0.000 |  0.059 |   0.075 |
| SchLocation     |     -0.011 |        0.002 |    -5.361 |      0.000 | -0.015 |  -0.007 |
| Gender          |     -0.011 |        0.002 |    -6.254 |      0.000 | -0.015 |  -0.008 |
| IMR             |     -0.019 |        0.002 |    -9.898 |      0.000 | -0.023 |  -0.015 |
| C(Grade)[T.CE2] |   

In [43]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'RegionCode'}
# 双向聚类标准误，同时聚类到地区和时间：vcov={'CRV3': 'RegionCode + Date'} 

try:
    model_result = feols(fml=RBformula3_2, data=df, vcov= 'hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())

except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: ExcellenceRate, Fixed effects: District^SchYear+Date
Inference:  hetero
Observations:  64743

| Coefficient     |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:----------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Normr1h         |     -0.010 |        0.003 |    -3.698 |      0.000 | -0.015 |  -0.005 |
| Normr3h         |      0.027 |        0.004 |     7.066 |      0.000 |  0.019 |   0.034 |
| NormFluctMonth  |      0.001 |        0.002 |     0.512 |      0.608 | -0.003 |   0.005 |
| SchType         |      0.043 |        0.005 |     8.243 |      0.000 |  0.032 |   0.053 |
| SchLocation     |      0.018 |        0.002 |     7.916 |      0.000 |  0.014 |   0.023 |
| Gender          |     -0.009 |        0.002 |    -4.648 |      0.000 | -0.013 |  -0.005 |
| IMR             |     -0.011 |        0.002 |    -5.256 |      0.000 | -0.015 |  -0.007 |
| C(Grade)[T.CE2

In [44]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'RegionCode'}
# 双向聚类标准误，同时聚类到地区和时间：vcov={'CRV3': 'RegionCode + Date'} 

try:
    model_result = feols(fml=RBformula4_1, data=df, vcov= 'hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())

except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: PassRate, Fixed effects: District^Date
Inference:  hetero
Observations:  64743

| Coefficient     |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:----------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Normr1h         |     -0.002 |        0.006 |    -0.299 |      0.765 | -0.014 |   0.011 |
| Normr3h         |      0.029 |        0.008 |     3.808 |      0.000 |  0.014 |   0.044 |
| NormFluctMonth  |      0.047 |        0.010 |     4.517 |      0.000 |  0.027 |   0.067 |
| SchType         |      0.067 |        0.004 |    16.007 |      0.000 |  0.058 |   0.075 |
| SchLocation     |     -0.014 |        0.002 |    -6.471 |      0.000 | -0.018 |  -0.010 |
| Gender          |     -0.011 |        0.002 |    -6.294 |      0.000 | -0.015 |  -0.008 |
| IMR             |     -0.019 |        0.002 |    -9.725 |      0.000 | -0.023 |  -0.015 |
| C(Grade)[T.CE2] |      0.011

In [45]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'RegionCode'}
# 双向聚类标准误，同时聚类到地区和时间：vcov={'CRV3': 'RegionCode + Date'} 

try:
    model_result = feols(fml=RBformula4_2, data=df, vcov= 'hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())

except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: ExcellenceRate, Fixed effects: District^Date
Inference:  hetero
Observations:  64743

| Coefficient     |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:----------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Normr1h         |     -0.028 |        0.006 |    -4.451 |      0.000 | -0.040 |  -0.016 |
| Normr3h         |      0.054 |        0.008 |     6.855 |      0.000 |  0.039 |   0.070 |
| NormFluctMonth  |      0.022 |        0.010 |     2.262 |      0.024 |  0.003 |   0.042 |
| SchType         |      0.043 |        0.005 |     8.356 |      0.000 |  0.033 |   0.054 |
| SchLocation     |      0.015 |        0.002 |     6.655 |      0.000 |  0.011 |   0.020 |
| Gender          |     -0.009 |        0.002 |    -4.671 |      0.000 | -0.013 |  -0.005 |
| IMR             |     -0.014 |        0.002 |    -6.205 |      0.000 | -0.018 |  -0.009 |
| C(Grade)[T.CE2] |     

# 二、调节效应（SchType, SchLocation, Gender设计交互项）

In [46]:
# 构建回归公式
fe_part = 'District + Date'
fe_part2 = 'RegionCode + Date'
fe_part3 = 'District^SchYear+Date'

control_vars = 'IMR + C(Grade)'
Moderator_vars = 'SchType + SchLocation + Gender'

# r1h * (A + B + C) 等价于:
# r1h + A + B + C + r1h:A + r1h:B + r1h:C


Moderationformula_1 = f"PassRate ~ (Normr1h + Normr3h + NormFluctMonth)*({Moderator_vars}) + {control_vars} | {fe_part}"
Moderationformula_2 = f"ExcellenceRate ~ (Normr1h + Normr3h + NormFluctMonth)*({Moderator_vars}) + {control_vars} | {fe_part}"


#鲁棒性检验
RBModerationformula_model5 = f"PassRate ~ (NormWinsr1h + NormWinsr3h + NormWinsFluctMonth)*({Moderator_vars}) + {control_vars} | {fe_part}"
RBModerationformula_model6 = f"ExcellenceRate ~ (NormWinsr1h + NormWinsr3h + NormWinsFluctMonth)*({Moderator_vars}) + {control_vars} | {fe_part}"
RBModerationformula_model7 = f"PassRate ~ (NormWinsr1h2 + NormWinsr3h2 + NormWinsFluctMonth2)*({Moderator_vars}) + {control_vars} | {fe_part}"
RBModerationformula_model8 = f"ExcellenceRate ~ (NormWinsr1h2 + NormWinsr3h2 + NormWinsFluctMonth2)*({Moderator_vars}) + {control_vars} | {fe_part}"

RBModerationformula_model13 = f"PassRate ~ (Normr1h + Normr3h + NormFluctMonth)*({Moderator_vars}) + {control_vars} | {fe_part2}"
RBModerationformula_model14 = f"ExcellenceRate ~ (Normr1h + Normr3h + NormFluctMonth)*({Moderator_vars}) + {control_vars} | {fe_part2}"

In [47]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'ADM1_PCODE'}


try:
    model_result = feols(fml=Moderationformula_1, data=df, vcov='hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())


except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: PassRate, Fixed effects: District+Date
Inference:  hetero
Observations:  64743

| Coefficient                |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:---------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Normr1h                    |     -0.011 |        0.003 |    -3.831 |      0.000 | -0.017 |  -0.006 |
| Normr3h                    |      0.025 |        0.004 |     7.240 |      0.000 |  0.018 |   0.032 |
| NormFluctMonth             |     -0.024 |        0.002 |   -11.205 |      0.000 | -0.028 |  -0.020 |
| SchType                    |      0.064 |        0.004 |    15.103 |      0.000 |  0.055 |   0.072 |
| SchLocation                |     -0.009 |        0.002 |    -4.470 |      0.000 | -0.013 |  -0.005 |
| Gender                     |     -0.011 |        0.002 |    -6.216 |      0.000 | -0.015 |  -0.008 |
| IMR                        |    

In [48]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'ADM1_PCODE'}


try:
    model_result = feols(fml=Moderationformula_2, data=df, vcov='hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())


except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: ExcellenceRate, Fixed effects: District+Date
Inference:  hetero
Observations:  64743

| Coefficient                |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:---------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Normr1h                    |     -0.018 |        0.003 |    -5.603 |      0.000 | -0.024 |  -0.011 |
| Normr3h                    |      0.031 |        0.004 |     8.287 |      0.000 |  0.024 |   0.039 |
| NormFluctMonth             |     -0.017 |        0.002 |    -8.069 |      0.000 | -0.021 |  -0.013 |
| SchType                    |      0.040 |        0.005 |     7.192 |      0.000 |  0.029 |   0.050 |
| SchLocation                |      0.019 |        0.002 |     8.437 |      0.000 |  0.015 |   0.024 |
| Gender                     |     -0.009 |        0.002 |    -4.619 |      0.000 | -0.013 |  -0.005 |
| IMR                       

In [49]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'ADM1_PCODE'}


try:
    model_result = feols(fml=RBModerationformula_model5, data=df, vcov='hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())


except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: PassRate, Fixed effects: District+Date
Inference:  hetero
Observations:  64743

| Coefficient                    |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:-------------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| NormWinsr1h                    |     -0.011 |        0.003 |    -3.805 |      0.000 | -0.017 |  -0.005 |
| NormWinsr3h                    |      0.025 |        0.004 |     7.250 |      0.000 |  0.019 |   0.032 |
| NormWinsFluctMonth             |     -0.024 |        0.002 |   -11.247 |      0.000 | -0.028 |  -0.020 |
| SchType                        |      0.064 |        0.004 |    15.106 |      0.000 |  0.056 |   0.072 |
| SchLocation                    |     -0.009 |        0.002 |    -4.468 |      0.000 | -0.013 |  -0.005 |
| Gender                         |     -0.011 |        0.002 |    -6.217 |      0.000 | -0.015 |  -0.008 |
| 

In [50]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'ADM1_PCODE'}


try:
    model_result = feols(fml=RBModerationformula_model6, data=df, vcov='hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())


except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: ExcellenceRate, Fixed effects: District+Date
Inference:  hetero
Observations:  64743

| Coefficient                    |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:-------------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| NormWinsr1h                    |     -0.018 |        0.003 |    -5.539 |      0.000 | -0.024 |  -0.011 |
| NormWinsr3h                    |      0.031 |        0.004 |     8.262 |      0.000 |  0.024 |   0.039 |
| NormWinsFluctMonth             |     -0.017 |        0.002 |    -8.076 |      0.000 | -0.021 |  -0.013 |
| SchType                        |      0.040 |        0.005 |     7.197 |      0.000 |  0.029 |   0.050 |
| SchLocation                    |      0.019 |        0.002 |     8.436 |      0.000 |  0.015 |   0.024 |
| Gender                         |     -0.009 |        0.002 |    -4.619 |      0.000 | -0.013 |  -0.00

In [51]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'ADM1_PCODE'}


try:
    model_result = feols(fml=RBModerationformula_model7, data=df, vcov='hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())


except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: PassRate, Fixed effects: District+Date
Inference:  hetero
Observations:  64743

| Coefficient                     |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| NormWinsr1h2                    |     -0.005 |        0.003 |    -1.784 |      0.074 | -0.011 |   0.001 |
| NormWinsr3h2                    |      0.012 |        0.004 |     3.125 |      0.002 |  0.004 |   0.019 |
| NormWinsFluctMonth2             |     -0.013 |        0.003 |    -5.239 |      0.000 | -0.018 |  -0.008 |
| SchType                         |      0.063 |        0.004 |    15.165 |      0.000 |  0.055 |   0.071 |
| SchLocation                     |     -0.009 |        0.002 |    -4.313 |      0.000 | -0.013 |  -0.005 |
| Gender                          |     -0.011 |        0.002 |    -6.205 |      0.000 | -0.015 |  -0.

In [52]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'ADM1_PCODE'}


try:
    model_result = feols(fml=RBModerationformula_model8, data=df, vcov='hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())


except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: ExcellenceRate, Fixed effects: District+Date
Inference:  hetero
Observations:  64743

| Coefficient                     |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| NormWinsr1h2                    |     -0.012 |        0.003 |    -3.606 |      0.000 | -0.019 |  -0.006 |
| NormWinsr3h2                    |      0.022 |        0.004 |     5.378 |      0.000 |  0.014 |   0.030 |
| NormWinsFluctMonth2             |     -0.010 |        0.003 |    -3.685 |      0.000 | -0.015 |  -0.005 |
| SchType                         |      0.041 |        0.005 |     7.676 |      0.000 |  0.031 |   0.052 |
| SchLocation                     |      0.020 |        0.002 |     8.526 |      0.000 |  0.015 |   0.024 |
| Gender                          |     -0.009 |        0.002 |    -4.615 |      0.000 | -0.013 

In [53]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'ADM1_PCODE'}


try:
    model_result = feols(fml=RBModerationformula_model13, data=df, vcov='hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())


except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: PassRate, Fixed effects: RegionCode+Date
Inference:  hetero
Observations:  64743

| Coefficient                |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:---------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Normr1h                    |     -0.013 |        0.003 |    -4.235 |      0.000 | -0.018 |  -0.007 |
| Normr3h                    |      0.018 |        0.004 |     4.555 |      0.000 |  0.010 |   0.026 |
| NormFluctMonth             |     -0.026 |        0.002 |   -11.988 |      0.000 | -0.030 |  -0.021 |
| SchType                    |      0.063 |        0.004 |    14.785 |      0.000 |  0.054 |   0.071 |
| SchLocation                |     -0.011 |        0.002 |    -5.214 |      0.000 | -0.015 |  -0.007 |
| Gender                     |     -0.011 |        0.002 |    -6.220 |      0.000 | -0.015 |  -0.008 |
| IMR                        |  

In [54]:
#运行回归 (使用 feols)
# 使用 feols (Fixed Effects Ordinary Least Squares) 运行模型
# 计量研究通常需要聚类标准误(Cluster SE)或者稳健标准误(Hetero Robust)
# vcov='hetero' 对应 Stata 的 robust 选项 (HC1)
# 如果需要聚类标准误到地区层面，可以使用 vcov={'CRV1': 'ADM1_PCODE'}


try:
    model_result = feols(fml=RBModerationformula_model14, data=df, vcov='hetero')

    print("\n--- 回归结果 (PyFixest OLS) ---")
    print(model_result.summary())


except Exception as e:
    print(f"\n运行回归时发生错误：{e}")


--- 回归结果 (PyFixest OLS) ---
###

Estimation:  OLS
Dep. var.: ExcellenceRate, Fixed effects: RegionCode+Date
Inference:  hetero
Observations:  64743

| Coefficient                |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:---------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Normr1h                    |     -0.019 |        0.003 |    -5.868 |      0.000 | -0.025 |  -0.012 |
| Normr3h                    |      0.027 |        0.004 |     6.373 |      0.000 |  0.019 |   0.036 |
| NormFluctMonth             |     -0.016 |        0.002 |    -7.480 |      0.000 | -0.020 |  -0.012 |
| SchType                    |      0.038 |        0.006 |     6.875 |      0.000 |  0.027 |   0.049 |
| SchLocation                |      0.018 |        0.002 |     7.894 |      0.000 |  0.014 |   0.023 |
| Gender                     |     -0.009 |        0.002 |    -4.618 |      0.000 | -0.013 |  -0.005 |
| IMR                     